# 🇮🇳 LangGraph Agentic English → Hindi Translation System

A production-grade agentic AI translation pipeline featuring:
- **LangGraph** nodes + conditional retry loop
- **Three-tier memory**: Working · Episodic · Semantic (ChromaDB)
- **LaBSE** multilingual embeddings for semantic search
- **Four evaluation metrics**: BLEU · chrF · BERTScore · COMET
- **Fine-tuning** pipeline with QLoRA
- Iterative self-improvement via Evaluator → Feedback → Retry

---
**Runtime:** GPU (T4 recommended) · **Estimated setup time:** ~5 min

## Step 1 — Install Dependencies

In [ ]:
# Core LangGraph + LangChain stack
!pip install -q langgraph>=0.2.0 langchain>=0.3.0 langchain-core>=0.3.0 langchain-openai>=0.2.0

# Embedding model
!pip install -q sentence-transformers>=3.0.0

# Vector database
!pip install -q chromadb>=0.5.0

# Episodic memory store
!pip install -q sqlitedict>=2.1.0

# Evaluation metrics
!pip install -q sacrebleu>=2.4.0 bert-score>=0.3.13

# Fine-tuning stack (HuggingFace + PEFT + TRL + quantisation)
!pip install -q transformers>=4.44.0 datasets>=2.21.0 peft>=0.12.0 trl>=0.10.0 bitsandbytes>=0.43.0 accelerate>=0.34.0

# Utilities
!pip install -q loguru typing_extensions

# COMET (optional — needs GPU, large download; comment out if not needed)
# !pip install -q unbabel-comet>=2.2.0
!pip install -q langchain-groq

print("✅ All dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00
✅ All dependencies installed


## Step 2 — API Key & Configuration

In [ ]:
import os
from getpass import getpass

# Get free key from: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

os.environ["TRANSLATION_MODEL"] = "llama-3.3-70b-versatile"
os.environ["PASS_THRESHOLD"]    = "0.60"
os.environ["W_BLEU"]  = "0.20"
os.environ["W_CHRF"]  = "0.25"
os.environ["W_BERT"]  = "0.30"
os.environ["W_COMET"] = "0.25"

print("✅ Configuration set")
print(f"   Model    : {os.environ['TRANSLATION_MODEL']}")
print(f"   Threshold: {os.environ['PASS_THRESHOLD']}")

Enter your Groq API key: ··········
✅ Configuration set
   Model    : llama-3.3-70b-versatile
   Threshold: 0.60


## Step 3 — State Schema

The `TranslationState` TypedDict is the **shared graph state** — every LangGraph node reads from it and returns partial updates that LangGraph merges automatically.

In [ ]:
from __future__ import annotations

from typing import Annotated, Any
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


class EvalScores(TypedDict, total=False):
    """Metric snapshot produced by the Evaluator node."""
    bleu: float           # sacrebleu sentence BLEU  (0–100)
    chrf: float           # chrF score               (0–100)
    bert_score_f1: float  # BERTScore F1             (0–1)
    comet: float          # COMET DA quality score   (0–1, normalised)
    aggregate: float      # weighted composite       (0–1)


class TranslationState(TypedDict, total=False):
    """Complete graph state — flows through every node."""

    # Input
    source_text: str
    session_id: str

    # Memory
    working_memory: dict[str, Any]       # volatile, session-scoped
    episodic_hits: list[dict[str, Any]]  # retrieved past episodes
    semantic_hits: list[dict[str, Any]]  # retrieved knowledge chunks

    # Embeddings
    source_embedding: list[float]

    # Translation
    translation: str
    iteration: int
    max_iterations: int

    # Evaluation
    scores: EvalScores
    reference_translations: list[str]
    passed: bool

    # Feedback
    feedback: str
    score_history: list[EvalScores]

    # Output
    final_translation: str
    final_scores: EvalScores

    # LangChain message log (optional multi-turn support)
    messages: Annotated[list, add_messages]


print("✅ State schema defined")
print(f"   Fields: {list(TranslationState.__annotations__.keys())}")

✅ State schema defined
   Fields: ['source_text', 'session_id', 'working_memory', 'episodic_hits', 'semantic_hits', 'source_embedding', 'translation', 'iteration', 'max_iterations', 'scores', 'reference_translations', 'passed', 'feedback', 'score_history', 'final_translation', 'final_scores', 'messages']


## Step 4 — Three-Tier Memory System

| Tier | Backend | Scope | Purpose |
|---|---|---|---|
| **Working** | Python dict | Session | Rolling context window, domain hints |
| **Episodic** | SQLiteDict (disk) | Persistent | Past episodes retrieved by cosine sim |
| **Semantic** | ChromaDB (HNSW) | Persistent | Grammar rules, idioms, phrase pairs |

In [ ]:
# ─── 4a. Embedding Model (LaBSE — Language-Agnostic BERT Sentence Embeddings) ───

from sentence_transformers import SentenceTransformer

_EMBED_MODEL_NAME = "sentence-transformers/LaBSE"
_embed_model_instance = None

def get_embedding_model() -> SentenceTransformer:
    """Singleton: load once, reuse everywhere."""
    global _embed_model_instance
    if _embed_model_instance is None:
        print(f"Loading embedding model: {_EMBED_MODEL_NAME} ...")
        _embed_model_instance = SentenceTransformer(_EMBED_MODEL_NAME)
        print("✅ Embedding model loaded")
    return _embed_model_instance


def encode_text(text):
    """
    Encode a string or list of strings into L2-normalised embeddings.
    Returns list[float] for a single string, list[list[float]] for a list.
    """
    model = get_embedding_model()
    result = model.encode(text, normalize_embeddings=True)
    if isinstance(text, str):
        return result.tolist()
    return [r.tolist() for r in result]


# Pre-load the model now (takes ~30 sec the first time)
_ = get_embedding_model()

Loading embedding model: sentence-transformers/LaBSE ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded


In [ ]:
# ─── 4b. Working Memory ───────────────────────────────────────────────────────

import time
from typing import Any


class WorkingMemory:
    """
    Volatile, in-process memory scoped to the current session.
    Keeps a rolling window of recent (source, translation) pairs
    so the translator LLM stays contextually consistent.
    """

    def __init__(self, session_id: str):
        self.session_id = session_id
        self._store: dict[str, Any] = {
            "session_id": session_id,
            "history": [],        # list of {source, translation, scores, timestamp}
            "domain_hint": None,  # e.g. "legal", "medical", "casual"
            "created_at": time.time(),
        }

    def add_translation(self, source: str, translation: str, scores: dict) -> None:
        """Append a completed translation; keep only the last 10."""
        self._store["history"].append({
            "source": source,
            "translation": translation,
            "scores": scores,
            "timestamp": time.time(),
        })
        self._store["history"] = self._store["history"][-10:]

    def set_domain(self, domain: str) -> None:
        self._store["domain_hint"] = domain

    def get_recent(self, n: int = 3) -> list[dict]:
        """Return last n (source, translation) pairs for in-context examples."""
        return self._store["history"][-n:]

    def to_dict(self) -> dict:
        return dict(self._store)


print("✅ WorkingMemory class defined")

✅ WorkingMemory class defined


In [ ]:
# ─── 4c. Episodic Memory (SQLiteDict — persists across Colab sessions) ────────

import json
import uuid
import numpy as np
from pathlib import Path
from sqlitedict import SqliteDict

DATA_DIR = Path("/content/translation_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
EPISODIC_DB_PATH = DATA_DIR / "episodic.sqlite"


class EpisodicMemory:
    """
    Disk-persisted store of completed translation episodes.
    Each episode stores: source, translation, scores, feedback,
    session_id, and the LaBSE embedding of the source text.

    Retrieval uses brute-force cosine similarity (fine for < 10k episodes).
    For larger corpora, replace with FAISS or a second ChromaDB collection.
    """

    def __init__(self):
        self._db = SqliteDict(str(EPISODIC_DB_PATH), autocommit=True)

    def store(
        self,
        source: str,
        translation: str,
        scores: dict,
        feedback: str,
        session_id: str,
        embedding: list[float],
    ) -> str:
        """Persist one episode and return its UUID."""
        episode_id = str(uuid.uuid4())
        self._db[episode_id] = json.dumps({
            "id": episode_id,
            "source": source,
            "translation": translation,
            "scores": scores,
            "feedback": feedback,
            "session_id": session_id,
            "embedding": embedding,
            "timestamp": time.time(),
        })
        return episode_id

    def retrieve_similar(
        self,
        query_embedding: list[float],
        top_k: int = 3,
        min_score: float = 0.5,
    ) -> list[dict]:
        """
        Return the top_k most similar past episodes (cosine similarity).
        Only includes episodes with similarity >= min_score.
        """
        q = np.array(query_embedding)
        results = []
        for raw in self._db.values():
            record = json.loads(raw)
            sim = float(np.dot(q, np.array(record["embedding"])))
            if sim >= min_score:
                results.append({**record, "_similarity": sim})
        results.sort(key=lambda x: x["_similarity"], reverse=True)
        return results[:top_k]

    def count(self) -> int:
        return len(self._db)

    def close(self):
        self._db.close()


print("✅ EpisodicMemory class defined")
print(f"   DB path: {EPISODIC_DB_PATH}")

✅ EpisodicMemory class defined
   DB path: /content/translation_data/episodic.sqlite


In [ ]:
# ─── 4d. Semantic Memory (ChromaDB vector store) ─────────────────────────────

import chromadb
from chromadb.config import Settings

CHROMA_DIR = DATA_DIR / "chroma_db"


class SemanticMemory:
    """
    ChromaDB-backed knowledge store with HNSW cosine index.

    Pre-seeded with Hindi grammar rules, idioms, register guidance,
    and domain glossaries. Grows automatically as high-quality
    translation pairs are added after each successful episode.
    """

    def __init__(self):
        self._client = chromadb.PersistentClient(
            path=str(CHROMA_DIR),
            settings=Settings(anonymized_telemetry=False),
        )
        self._collection = self._client.get_or_create_collection(
            name="hindi_translation_knowledge",
            metadata={"hnsw:space": "cosine"},
        )

    def add_knowledge(
        self,
        chunks: list[str],
        metadatas: list[dict] | None = None,
        ids: list[str] | None = None,
    ) -> None:
        """Embed chunks and upsert into ChromaDB."""
        if not chunks:
            return
        embeddings = encode_text(chunks)  # list of lists
        ids = ids or [str(uuid.uuid4()) for _ in chunks]
        metadatas = metadatas or [{} for _ in chunks]
        self._collection.upsert(
            documents=chunks,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids,
        )

    def retrieve(
        self,
        query_embedding: list[float],
        top_k: int = 5,
    ) -> list[dict]:
        """Return top_k most semantically similar knowledge chunks."""
        if self._collection.count() == 0:
            return []
        results = self._collection.query(
            query_embeddings=[query_embedding],
            n_results=min(top_k, self._collection.count()),
            include=["documents", "metadatas", "distances"],
        )
        return [
            {"chunk": doc, "metadata": meta, "distance": dist}
            for doc, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]

    def count(self) -> int:
        return self._collection.count()

    def seed_default_knowledge(self) -> None:
        """Populate starter Hindi translation knowledge if collection is empty."""
        if self._collection.count() > 0:
            print(f"   Semantic memory already has {self._collection.count()} chunks — skipping seed")
            return

        seed_chunks = [
            # Grammar
            "Hindi verbs agree with subject in gender and number. Masculine: करता है. Feminine: करती है.",
            "Hindi uses postpositions (परसर्ग) after nouns instead of prepositions. 'In the house' = घर में.",
            "The standard Hindi sentence order is Subject-Object-Verb (SOV), unlike English SVO.",
            "Plural in Hindi: add -एँ for feminine nouns (लड़की → लड़कियाँ), -ें/-ों for oblique cases.",
            # Idioms
            "English idiom 'break the ice' translates to 'बात की शुरुआत करना' in Hindi.",
            "English 'once in a blue moon' → Hindi 'कभी-कभार' (rarely, occasionally).",
            "'It's raining cats and dogs' → Hindi 'मूसलाधार बारिश हो रही है' (torrential rain).",
            # Register
            "Formal Hindi 'आप' (you) vs informal 'तुम' vs very informal 'तू'. Match English register.",
            "In formal Hindi, use 'कृपया' for please, 'धन्यवाद' for thank you, 'क्षमा करें' for excuse me.",
            # Common phrases
            "Good morning = सुप्रभात (formal). Good night = शुभ रात्रि. Good evening = शुभ संध्या.",
            "Thank you = धन्यवाद (formal) or शुक्रिया (informal). You're welcome = आपका स्वागत है.",
            "How are you? = आप कैसे हैं? (formal) / तुम कैसे हो? (informal). I am fine = मैं ठीक हूँ.",
            # Technical
            "Technical terms: software = सॉफ़्टवेयर, computer = कंप्यूटर, internet = इंटरनेट, mobile = मोबाइल.",
            "Medical: hospital = अस्पताल, doctor = डॉक्टर/चिकित्सक, medicine = दवाई, patient = मरीज़.",
            # Numbers and time
            "Hindi numbers: 1=एक, 2=दो, 3=तीन, 4=चार, 5=पाँच, 10=दस, 100=सौ, 1000=हज़ार, 100000=लाख.",
            "Time: morning = सुबह, afternoon = दोपहर, evening = शाम, night = रात. Today = आज, tomorrow = कल.",
        ]

        metadatas = [
            {"type": "grammar"}, {"type": "grammar"}, {"type": "grammar"}, {"type": "grammar"},
            {"type": "idiom"}, {"type": "idiom"}, {"type": "idiom"},
            {"type": "register"}, {"type": "register"},
            {"type": "phrase"}, {"type": "phrase"}, {"type": "phrase"},
            {"type": "technical"}, {"type": "technical"},
            {"type": "vocabulary"}, {"type": "vocabulary"},
        ]

        self.add_knowledge(seed_chunks, metadatas)
        print(f"✅ Seeded {len(seed_chunks)} knowledge chunks into SemanticMemory")


print("✅ SemanticMemory class defined")

✅ SemanticMemory class defined


In [ ]:
# ─── 4e. Initialise all three memory systems ──────────────────────────────────

SESSION_ID = str(uuid.uuid4())

working_memory  = WorkingMemory(SESSION_ID)
episodic_memory = EpisodicMemory()
semantic_memory = SemanticMemory()
semantic_memory.seed_default_knowledge()

print(f"\n✅ Memory systems ready")
print(f"   Session ID        : {SESSION_ID}")
print(f"   Episodic episodes : {episodic_memory.count()}")
print(f"   Semantic chunks   : {semantic_memory.count()}")

   Semantic memory already has 16 chunks — skipping seed

✅ Memory systems ready
   Session ID        : 5ba65b45-efda-4307-a478-52697d5ea00a
   Episodic episodes : 0
   Semantic chunks   : 16


## Step 5 — Evaluation Metrics

Four complementary metrics scored on every translation attempt:

| Metric | Range | Weight | Needs references? |
|---|---|---|---|
| BLEU | 0–100 | 20% | Yes |
| chrF | 0–100 | 25% | Yes |
| BERTScore F1 | 0–1 | 30% | Optional |
| COMET DA | 0–1 | 25% | No (QE mode) |

In [ ]:
# ─── Evaluation metrics ────────────────────────────────────────────────────────

from typing import Optional

# Metric weights from env
METRIC_WEIGHTS = {
    "bleu":          float(os.getenv("W_BLEU",  "0.20")),
    "chrf":          float(os.getenv("W_CHRF",  "0.25")),
    "bert_score_f1": float(os.getenv("W_BERT",  "0.30")),
    "comet":         float(os.getenv("W_COMET", "0.25")),
}


# ── BLEU ─────────────────────────────────────────────────────────────────────
def compute_bleu(hypothesis: str, references: list[str]) -> float:
    """Sentence-level BLEU via sacrebleu. Returns 0 if no references."""
    if not references:
        return 0.0
    try:
        from sacrebleu.metrics import BLEU
        return float(BLEU(effective_order=True).sentence_score(hypothesis, references).score)
    except Exception as e:
        print(f"[BLEU] Warning: {e}")
        return 0.0


# ── chrF ─────────────────────────────────────────────────────────────────────
def compute_chrf(hypothesis: str, references: list[str]) -> float:
    """
    Character-level F-score via sacrebleu.
    More robust than BLEU for Devanagari script.
    Returns 0 if no references.
    """
    if not references:
        return 0.0
    try:
        from sacrebleu.metrics import CHRF
        return float(CHRF().sentence_score(hypothesis, references).score)
    except Exception as e:
        print(f"[chrF] Warning: {e}")
        return 0.0


# ── BERTScore ────────────────────────────────────────────────────────────────
def _devanagari_heuristic(text: str) -> float:
    """Reference-free proxy: fraction of Devanagari characters × length penalty."""
    if not text:
        return 0.0
    devanagari = sum(1 for c in text if "\u0900" <= c <= "\u097F")
    script_ratio = min(devanagari / max(len(text), 1) * 1.2, 1.0)
    length_score = min(len(text.split()) / 3.0, 1.0)
    return round(script_ratio * 0.7 + length_score * 0.3, 4)


def compute_bert_score(
    hypothesis: str,
    references: list[str],
    model_type: str = "bert-base-multilingual-cased",
) -> float:
    """BERTScore F1 with multilingual BERT. Falls back to heuristic if no references."""
    if not references:
        return _devanagari_heuristic(hypothesis)
    try:
        from bert_score import score as bert_score_fn
        _, _, F1 = bert_score_fn(
            cands=[hypothesis],
            refs=[references[0]],
            model_type=model_type,
            lang="hi",
            verbose=False,
        )
        return float(F1[0])
    except Exception as e:
        print(f"[BERTScore] Warning: {e} — using heuristic")
        return _devanagari_heuristic(hypothesis)


# ── COMET ────────────────────────────────────────────────────────────────────
def _comet_heuristic(hypothesis: str) -> float:
    """Lightweight COMET fallback (no GPU needed)."""
    if not hypothesis:
        return 0.0
    devanagari = sum(1 for c in hypothesis if "\u0900" <= c <= "\u097F")
    script_score = min(devanagari / max(len(hypothesis), 1) * 1.2, 1.0)
    length_score = min(len(hypothesis.split()) / 5.0, 1.0)
    return round(script_score * 0.7 + length_score * 0.3, 4)


def compute_comet(
    source: str,
    hypothesis: str,
    references: Optional[list[str]] = None,
    comet_model_name: str = "Unbabel/wmt22-comet-da",
) -> float:
    """
    COMET quality estimation (reference-free DA variant).
    Falls back to heuristic if unbabel-comet is not installed.
    Normalised from [-1, 1] to [0, 1].
    """
    try:
        from comet import download_model, load_from_checkpoint
        model_path = download_model(comet_model_name)
        model = load_from_checkpoint(model_path)
        data = [{"src": source, "mt": hypothesis}]
        if references:
            data[0]["ref"] = references[0]
        raw = model.predict(data, batch_size=1, gpus=0).scores[0]
        return round(float((raw + 1.0) / 2.0), 4)  # normalise to [0, 1]
    except ImportError:
        return _comet_heuristic(hypothesis)
    except Exception as e:
        print(f"[COMET] Warning: {e} — using heuristic")
        return _comet_heuristic(hypothesis)


# ── Composite scorer ─────────────────────────────────────────────────────────
def compute_all_metrics(
    hypothesis: str,
    references: list[str],
    source: str,
) -> dict:
    """Run all four metrics. Failures degrade gracefully (return 0)."""
    return {
        "bleu":          round(compute_bleu(hypothesis, references),               2),
        "chrf":          round(compute_chrf(hypothesis, references),               2),
        "bert_score_f1": round(compute_bert_score(hypothesis, references),         4),
        "comet":         round(compute_comet(source, hypothesis, references),      4),
    }


def aggregate_score(scores: dict) -> float:
    """
    Weighted composite [0, 1].
    When references are absent (BLEU + chrF = 0), shift weight to BERT + COMET.
    """
    bleu_n  = scores.get("bleu",          0.0) / 100.0
    chrf_n  = scores.get("chrf",          0.0) / 100.0
    bert_f1 = scores.get("bert_score_f1", 0.0)
    comet   = scores.get("comet",         0.0)

    has_refs = bleu_n > 0 or chrf_n > 0
    w = METRIC_WEIGHTS if has_refs else {"bleu": 0, "chrf": 0, "bert_score_f1": 0.5, "comet": 0.5}

    return round(
        w["bleu"]          * bleu_n
        + w["chrf"]          * chrf_n
        + w["bert_score_f1"] * bert_f1
        + w["comet"]         * comet,
        4,
    )


def format_score_table(score_history: list[dict]) -> str:
    """ASCII comparison table across all iterations."""
    if not score_history:
        return "No score history available."
    rows = ["Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate"]
    rows.append("─────┼────────┼────────┼───────────┼─────────┼──────────")
    for i, s in enumerate(score_history, 1):
        rows.append(
            f"  {i:2d} │ {s.get('bleu',0):6.2f} │ {s.get('chrf',0):6.2f} │"
            f"   {s.get('bert_score_f1',0):.4f}  │ {s.get('comet',0):.4f}  │"
            f"  {s.get('aggregate',0):.4f}"
        )
    return "\n".join(rows)


# Quick smoke test
test_scores = compute_all_metrics(
    hypothesis="नमस्ते, आप कैसे हैं?",
    references=["नमस्ते, आप कैसे हैं?"],
    source="Hello, how are you?",
)
test_scores["aggregate"] = aggregate_score(test_scores)
print("✅ Metrics defined. Smoke-test scores:")
for k, v in test_scores.items():
    print(f"   {k:20s}: {v}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Metrics defined. Smoke-test scores:
   bleu                : 100.0
   chrf                : 100.0
   bert_score_f1       : 1.0
   comet               : 0.87
   aggregate           : 0.9675


## Step 6 — LangGraph Node Functions

Each node receives `TranslationState` and returns a **partial dict** that LangGraph merges into the shared state.

In [ ]:
# ─── LLM helper (Groq) ────────────────────────────────────────────────────────

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import textwrap

def get_llm(temperature: float = 0.3):
    return ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=temperature,
        max_tokens=512,
        api_key=os.environ["GROQ_API_KEY"],
    )

print("✅ LLM helper ready — model: llama-3.3-70b-versatile")

✅ LLM helper ready — model: llama-3.3-70b-versatile


In [ ]:
# ─── Node 1: Input ────────────────────────────────────────────────────────────

def input_node(state: TranslationState) -> dict:
    """
    Validates and normalises the incoming English text.
    Resets iteration counter and score history for a fresh run.
    """
    source = state.get("source_text", "").strip()
    if not source:
        raise ValueError("source_text must not be empty")

    print(f"\n{'='*60}")
    print(f"[Node 1 — Input] '{source[:70]}...'" if len(source) > 70 else f"[Node 1 — Input] '{source}'")

    return {
        "source_text":      source,
        "iteration":        0,
        "max_iterations":   state.get("max_iterations", 3),
        "score_history":    [],
        "passed":           False,
        "translation":      "",
        "final_translation": "",
    }


print("✅ input_node defined")

✅ input_node defined


In [ ]:
# ─── Node 2: Embedding + Memory Retrieval ────────────────────────────────────

from functools import partial


def embedding_node(
    state: TranslationState,
    working: WorkingMemory,
    episodic: EpisodicMemory,
    semantic: SemanticMemory,
) -> dict:
    """
    1. Encode source with LaBSE (768-dim, L2-normalised)
    2. Query episodic memory for similar past translations
    3. Query semantic ChromaDB for relevant knowledge chunks
    4. Load current working memory context
    """
    source = state["source_text"]

    # Embed
    embedding = encode_text(source)  # list[float]

    # Episodic retrieval
    episodic_hits = episodic.retrieve_similar(embedding, top_k=3, min_score=0.6)

    # Semantic retrieval
    semantic_hits = semantic.retrieve(embedding, top_k=5)

    print(f"[Node 2 — Embedding] dim={len(embedding)} | "
          f"episodic_hits={len(episodic_hits)} | semantic_hits={len(semantic_hits)}")

    return {
        "source_embedding": embedding,
        "episodic_hits":    episodic_hits,
        "semantic_hits":    semantic_hits,
        "working_memory":   working.to_dict(),
    }


print("✅ embedding_node defined")

✅ embedding_node defined


In [ ]:
# ─── Node 3: Translation (LLM) ───────────────────────────────────────────────

def translation_node(state: TranslationState) -> dict:
    """
    Calls the LLM with a richly-contextualised prompt.
    Injects: semantic knowledge, episodic examples, working memory,
    and feedback from the previous iteration (on retries).
    Temperature increases slightly on each retry for diversity.
    """
    source        = state["source_text"]
    iteration     = state.get("iteration", 0)
    feedback      = state.get("feedback", "")
    semantic_hits = state.get("semantic_hits", [])
    episodic_hits = state.get("episodic_hits", [])
    working_mem   = state.get("working_memory", {})

    # Build context sections
    knowledge_section = ""
    if semantic_hits:
        chunks = "\n".join(f"  • {h['chunk']}" for h in semantic_hits[:4])
        knowledge_section = f"\n\n[Relevant Hindi knowledge]\n{chunks}"

    examples_section = ""
    if episodic_hits:
        lines = [f"  EN: {ep['source']}\n  HI: {ep['translation']}" for ep in episodic_hits[:2]]
        examples_section = "\n\n[Similar past translations]\n" + "\n---\n".join(lines)

    wm_section = ""
    recent = working_mem.get("history", [])[-3:]
    if recent:
        wm_lines = [f"  EN: {r['source']}\n  HI: {r['translation']}" for r in recent]
        wm_section = "\n\n[Recent session translations]\n" + "\n".join(wm_lines)

    feedback_section = ""
    if iteration > 0 and feedback:
        feedback_section = f"\n\n[Previous attempt feedback — fix these issues]\n{feedback}"

    system_prompt = textwrap.dedent(f"""
        You are an expert English-to-Hindi translator with deep knowledge of:
        - Standard Hindi (Manak Hindi / खड़ी बोली) and Devanagari script
        - Register matching (formal ↔ informal)
        - Cultural equivalence (not just literal translation)
        - Correct gender/number agreement and postpositions

        RULES:
        1. Output ONLY the Hindi translation in Devanagari — no romanisation, no explanations.
        2. Preserve the register (formal/informal) of the source.
        3. Use natural, idiomatic Hindi. Avoid word-for-word literal translation.
        4. Transliterate untranslatable technical terms (e.g. software → सॉफ़्टवेयर).
        5. This is iteration {iteration + 1} of {state.get('max_iterations', 3)}.
        {knowledge_section}
        {examples_section}
        {wm_section}
        {feedback_section}
    """).strip()

    llm = get_llm(temperature=0.2 + 0.05 * iteration)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Translate to Hindi:\n\n{source}"),
    ])
    translation = response.content.strip()

    print(f"[Node 3 — Translation] Iteration {iteration + 1}: '{translation[:60]}...'" \
          if len(translation) > 60 else \
          f"[Node 3 — Translation] Iteration {iteration + 1}: '{translation}'")

    return {
        "translation": translation,
        "iteration":   iteration + 1,
    }


print("✅ translation_node defined")

✅ translation_node defined


In [ ]:
# ─── Node 4: Evaluator ───────────────────────────────────────────────────────

def evaluator_node(state: TranslationState) -> dict:
    """
    Scores the current translation with all four metrics.
    Sets state['passed'] = True if aggregate >= threshold OR max iterations reached.
    Appends scores to score_history for trend analysis.
    """
    translation = state["translation"]
    source      = state["source_text"]
    references  = state.get("reference_translations", [])
    iteration   = state.get("iteration", 1)
    max_iter    = state.get("max_iterations", 3)
    threshold   = float(os.getenv("PASS_THRESHOLD", "0.60"))

    scores = compute_all_metrics(translation, references, source)
    agg    = aggregate_score(scores)
    scores["aggregate"] = agg

    passed  = agg >= threshold or iteration >= max_iter
    history = list(state.get("score_history", []))
    history.append(dict(scores))

    status = "✅ PASSED" if passed else "🔄 RETRY"
    print(f"[Node 4 — Evaluator] {status} | agg={agg:.3f} (threshold={threshold}) "
          f"| BLEU={scores['bleu']:.1f} chrF={scores['chrf']:.1f} "
          f"BERT={scores['bert_score_f1']:.3f} COMET={scores['comet']:.3f}")

    return {
        "scores":       scores,
        "passed":       passed,
        "score_history": history,
    }


print("✅ evaluator_node defined")

✅ evaluator_node defined


In [ ]:
# ─── Node 5: Feedback ────────────────────────────────────────────────────────

def feedback_node(state: TranslationState) -> dict:
    """
    If the translation passed: returns a brief success summary.
    If it failed: calls the LLM as a quality critic to generate
    targeted, actionable feedback for the next translation attempt.
    """
    source      = state["source_text"]
    translation = state["translation"]
    scores      = state.get("scores", {})
    passed      = state.get("passed", False)

    if passed:
        feedback = (
            f"Translation accepted. Aggregate={scores.get('aggregate',0):.3f} "
            f"| BLEU={scores.get('bleu','N/A')} "
            f"| chrF={scores.get('chrf','N/A')} "
            f"| BERTScore={scores.get('bert_score_f1',0):.3f}"
        )
        print(f"[Node 5 — Feedback] {feedback}")
        return {"feedback": feedback}

    # Generate targeted critique
    score_str = (
        f"BLEU: {scores.get('bleu','N/A'):.1f}/100 | "
        f"chrF: {scores.get('chrf','N/A'):.1f}/100 | "
        f"BERTScore: {scores.get('bert_score_f1',0):.3f} | "
        f"Aggregate: {scores.get('aggregate',0):.3f}"
    )
    critic_prompt = textwrap.dedent(f"""
        You are a professional Hindi language quality assessor.
        Analyse the translation below and provide 2–4 specific, actionable issues.

        English source   : {source}
        Hindi translation: {translation}
        Metric scores    : {score_str}

        Check for:
        - Gender/number agreement errors
        - Wrong postpositions (में, पर, को, से, के लिए, etc.)
        - Unnatural or overly literal phrasing
        - Missing or wrong diacritics (मात्राएँ)
        - Register mismatch (formal/informal)
        - Semantic errors or omissions

        Output format (be concise):
        Issue 1: <description + correction>
        Issue 2: <description + correction>
        Overall: <one sentence — what matters most to fix>
    """).strip()

    llm = get_llm(temperature=0.1)
    feedback = llm.invoke([HumanMessage(content=critic_prompt)]).content.strip()

    print(f"[Node 5 — Feedback] Critique generated ({len(feedback)} chars)")
    return {"feedback": feedback}


print("✅ feedback_node defined")

✅ feedback_node defined


In [ ]:
# ─── Node 6: Memory Update ───────────────────────────────────────────────────

def memory_update_node(
    state: TranslationState,
    working: WorkingMemory,
    episodic: EpisodicMemory,
    semantic: SemanticMemory,
) -> dict:
    """
    Persists the completed episode into all three memory tiers:
    - Working  → append to session history (in-process)
    - Episodic → store to disk with embedding (always)
    - Semantic → upsert translation pair as knowledge (only if aggregate >= 0.70)
    """
    source      = state["source_text"]
    translation = state["translation"]
    scores      = state.get("scores", {})
    feedback    = state.get("feedback", "")
    session_id  = state.get("session_id", "default")
    embedding   = state.get("source_embedding", [])
    aggregate   = scores.get("aggregate", 0)

    # 1. Working memory (always)
    working.add_translation(source, translation, scores)

    # 2. Episodic memory (always, if embedding exists)
    if embedding:
        ep_id = episodic.store(source, translation, scores, feedback, session_id, embedding)

    # 3. Semantic memory (only for high-quality translations)
    if aggregate >= 0.70 and embedding:
        chunk = f"English: {source}\nHindi: {translation}"
        semantic.add_knowledge(
            chunks=[chunk],
            metadatas=[{"type": "translation_pair", "score": round(aggregate, 4)}],
        )
        print(f"[Node 6 — Memory] All 3 tiers updated (quality pair added to semantic)")
    else:
        print(f"[Node 6 — Memory] Working + Episodic updated (aggregate {aggregate:.3f} < 0.70 — skipping semantic)")

    return {"working_memory": working.to_dict()}


print("✅ memory_update_node defined")

✅ memory_update_node defined


In [ ]:
# ─── Node 7: Output ──────────────────────────────────────────────────────────

def output_node(state: TranslationState) -> dict:
    """
    Packages the final result:
    - final_translation: the best Hindi translation
    - final_scores: EvalScores from the last iteration
    - working_memory: updated with per-iteration comparison table
    """
    translation   = state["translation"]
    scores        = state.get("scores", {})
    score_history = state.get("score_history", [])

    comparison_report = format_score_table(score_history)

    print(f"[Node 7 — Output] Final translation: '{translation}'")
    print(f"\n{comparison_report}")

    return {
        "final_translation": translation,
        "final_scores":      scores,
        "working_memory": {
            **state.get("working_memory", {}),
            "_last_comparison_report": comparison_report,
        },
    }


# ─── Conditional edge ─────────────────────────────────────────────────────────

def should_retry(state: TranslationState) -> str:
    """
    LangGraph conditional edge function.
    'retry'   → route back to translation_node
    'proceed' → route forward to feedback_node
    """
    if state.get("passed", False):
        return "proceed"
    return "retry"


print("✅ output_node + should_retry defined")

✅ output_node + should_retry defined


## Step 7 — Build & Compile the LangGraph

Wire all 7 nodes together with edges and the conditional retry loop.

In [ ]:
from langgraph.graph import StateGraph, END
from functools import partial


def build_graph(
    working: WorkingMemory,
    episodic: EpisodicMemory,
    semantic: SemanticMemory,
):
    """
    Compile and return the LangGraph StateGraph.
    Memory objects are injected via functools.partial so node
    signatures remain (state: TranslationState) -> dict.
    """
    # Bind memory dependencies
    _embedding_node     = partial(embedding_node,     working=working, episodic=episodic, semantic=semantic)
    _memory_update_node = partial(memory_update_node, working=working, episodic=episodic, semantic=semantic)

    graph = StateGraph(TranslationState)

    # ── Add nodes ─────────────────────────────────────────────────────────────
    graph.add_node("input_node",         input_node)
    graph.add_node("embedding_node",     _embedding_node)
    graph.add_node("translation_node",   translation_node)
    graph.add_node("evaluator_node",     evaluator_node)
    graph.add_node("feedback_node",      feedback_node)
    graph.add_node("memory_update_node", _memory_update_node)
    graph.add_node("output_node",        output_node)

    # ── Normal flow edges ────────────────────────────────────────────────────
    graph.add_edge("input_node",         "embedding_node")
    graph.add_edge("embedding_node",     "translation_node")
    graph.add_edge("translation_node",   "evaluator_node")

    # ── Conditional edge: retry loop ─────────────────────────────────────────
    graph.add_conditional_edges(
        "evaluator_node",
        should_retry,
        {
            "retry":   "translation_node",   # ← loop back
            "proceed": "feedback_node",       # ← move forward
        },
    )

    graph.add_edge("feedback_node",      "memory_update_node")
    graph.add_edge("memory_update_node", "output_node")
    graph.add_edge("output_node",        END)

    graph.set_entry_point("input_node")
    return graph.compile()


# ── Compile the graph with our memory systems ─────────────────────────────────
app = build_graph(working_memory, episodic_memory, semantic_memory)

print("✅ LangGraph compiled successfully")
print("   Nodes:", ["input", "embedding", "translation", "evaluator",
                    "feedback", "memory_update", "output"])
print("   Conditional edge: evaluator → (retry → translation | proceed → feedback)")

✅ LangGraph compiled successfully
   Nodes: ['input', 'embedding', 'translation', 'evaluator', 'feedback', 'memory_update', 'output']
   Conditional edge: evaluator → (retry → translation | proceed → feedback)


## Step 8 — Run a Translation

Test the full pipeline end-to-end. Optionally provide human reference translations for BLEU/chrF scoring.

In [ ]:
# ─── High-level translate() helper ───────────────────────────────────────────

def translate(
    source_text: str,
    reference_translations: list[str] | None = None,
    max_iterations: int = 3,
    pass_threshold: float = 0.60,
) -> dict:
    """
    Translate an English sentence to Hindi using the LangGraph agent.

    Args:
        source_text             : English input
        reference_translations  : Optional human references for BLEU/chrF
        max_iterations          : Maximum retry attempts before accepting result
        pass_threshold          : Aggregate score cutoff (0–1)

    Returns dict with:
        final_translation, final_scores, score_history, feedback, iterations
    """
    os.environ["PASS_THRESHOLD"] = str(pass_threshold)

    initial_state = {
        "source_text":            source_text,
        "session_id":             SESSION_ID,
        "reference_translations": reference_translations or [],
        "max_iterations":         max_iterations,
    }

    final_state = app.invoke(initial_state)

    return {
        "final_translation": final_state.get("final_translation", ""),
        "final_scores":      final_state.get("final_scores",      {}),
        "score_history":     final_state.get("score_history",     []),
        "feedback":          final_state.get("feedback",          ""),
        "iterations":        final_state.get("iteration",         0),
    }


print("✅ translate() helper ready")

✅ translate() helper ready


In [ ]:
# ─── Test 1: Basic translation (no reference) ─────────────────────────────────

result1 = translate(
    source_text="Artificial intelligence is transforming the way we work and live.",
    max_iterations=3,
    pass_threshold=0.55,   # lower threshold for reference-free mode
)

print("\n" + "="*60)
print(f"ENGLISH  : Artificial intelligence is transforming the way we work and live.")
print(f"HINDI    : {result1['final_translation']}")
print(f"ITERS    : {result1['iterations']}")
print(f"AGG SCORE: {result1['final_scores'].get('aggregate', 0):.3f}")


[Node 1 — Input] 'Artificial intelligence is transforming the way we work and live.'
[Node 2 — Embedding] dim=768 | episodic_hits=0 | semantic_hits=5
[Node 3 — Translation] Iteration 1: 'कृत्रिम बुद्धिमत्ता हमारे काम करने और जीने के तरीके को बदल र...'
[Node 4 — Evaluator] ✅ PASSED | agg=0.987 (threshold=0.55) | BLEU=0.0 chrF=0.0 BERT=0.987 COMET=0.987
[Node 5 — Feedback] Translation accepted. Aggregate=0.987 | BLEU=0.0 | chrF=0.0 | BERTScore=0.987
[Node 6 — Memory] All 3 tiers updated (quality pair added to semantic)
[Node 7 — Output] Final translation: 'कृत्रिम बुद्धिमत्ता हमारे काम करने और जीने के तरीके को बदल रही है।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │   0.00 │   0.00 │   0.9873  │ 0.9873  │  0.9873

ENGLISH  : Artificial intelligence is transforming the way we work and live.
HINDI    : कृत्रिम बुद्धिमत्ता हमारे काम करने और जीने के तरीके को बदल रही है।
ITERS    : 1
AGG SCORE: 0.987


In [ ]:
# ─── Test 2: Translation with human reference (enables BLEU + chrF) ───────────

result2 = translate(
    source_text="Please submit your report by the end of the business day.",
    reference_translations=["कृपया कार्य दिवस के अंत तक अपनी रिपोर्ट जमा करें।"],
    max_iterations=3,
    pass_threshold=0.60,
)

print("\n" + "="*60)
print(f"ENGLISH  : Please submit your report by the end of the business day.")
print(f"HINDI    : {result2['final_translation']}")
print(f"REFERENCE: कृपया कार्य दिवस के अंत तक अपनी रिपोर्ट जमा करें।")
print(f"ITERS    : {result2['iterations']}")
print("\nScore breakdown:")
for k, v in result2['final_scores'].items():
    print(f"  {k:20s}: {v}")
print("\nIteration comparison:")
print(format_score_table(result2['score_history']))


[Node 1 — Input] 'Please submit your report by the end of the business day.'
[Node 2 — Embedding] dim=768 | episodic_hits=0 | semantic_hits=5
[Node 3 — Translation] Iteration 1: 'कृपया व्यावसायिक दिवस के अंत तक अपनी रिपोर्ट जमा करें।'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Node 4 — Evaluator] ✅ PASSED | agg=0.889 (threshold=0.6) | BLEU=78.2 chrF=79.7 BERT=0.943 COMET=1.000
[Node 5 — Feedback] Translation accepted. Aggregate=0.889 | BLEU=78.25 | chrF=79.72 | BERTScore=0.943
[Node 6 — Memory] All 3 tiers updated (quality pair added to semantic)
[Node 7 — Output] Final translation: 'कृपया व्यावसायिक दिवस के अंत तक अपनी रिपोर्ट जमा करें।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  78.25 │  79.72 │   0.9434  │ 1.0000  │  0.8888

ENGLISH  : Please submit your report by the end of the business day.
HINDI    : कृपया व्यावसायिक दिवस के अंत तक अपनी रिपोर्ट जमा करें।
REFERENCE: कृपया कार्य दिवस के अंत तक अपनी रिपोर्ट जमा करें।
ITERS    : 1

Score breakdown:
  bleu                : 78.25
  chrf                : 79.72
  bert_score_f1       : 0.9434
  comet               : 1.0
  aggregate           : 0.8888

Iteration comparison:
Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
──

In [ ]:
# ─── Test 3: Batch translation ────────────────────────────────────────────────

test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Good morning! How was your weekend?",
    "The meeting has been rescheduled to 3 PM tomorrow.",
    "Technology is making our lives easier every day.",
]

print("Running batch translation (this will make multiple API calls)...\n")
batch_results = []

for sentence in test_sentences:
    r = translate(sentence, max_iterations=2, pass_threshold=0.50)
    batch_results.append(r)
    print(f"EN: {sentence}")
    print(f"HI: {r['final_translation']}")
    print(f"   score={r['final_scores'].get('aggregate',0):.3f} | iters={r['iterations']}")
    print()

print(f"✅ Episodic memory now has {episodic_memory.count()} episodes")

Running batch translation (this will make multiple API calls)...


[Node 1 — Input] 'The quick brown fox jumps over the lazy dog.'
[Node 2 — Embedding] dim=768 | episodic_hits=0 | semantic_hits=5
[Node 3 — Translation] Iteration 1: 'तेज़ भूरा लोमड़ी आलसी कुत्ते पर कूदता है।'
[Node 4 — Evaluator] ✅ PASSED | agg=0.997 (threshold=0.5) | BLEU=0.0 chrF=0.0 BERT=0.997 COMET=0.997
[Node 5 — Feedback] Translation accepted. Aggregate=0.997 | BLEU=0.0 | chrF=0.0 | BERTScore=0.997
[Node 6 — Memory] All 3 tiers updated (quality pair added to semantic)
[Node 7 — Output] Final translation: 'तेज़ भूरा लोमड़ी आलसी कुत्ते पर कूदता है।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │   0.00 │   0.00 │   0.9966  │ 0.9966  │  0.9966
EN: The quick brown fox jumps over the lazy dog.
HI: तेज़ भूरा लोमड़ी आलसी कुत्ते पर कूदता है।
   score=0.997 | iters=1


[Node 1 — Input] 'Good morning! How was your weekend?'
[Node 2 — Embedding] dim=7

## Step 9 — Inspect Memory State

Examine what each memory tier has stored after the translations above.

In [ ]:
# ─── 9a. Working Memory ───────────────────────────────────────────────────────

print("=" * 60)
print("WORKING MEMORY — Session history")
print("=" * 60)
wm = working_memory.to_dict()
print(f"Session ID : {wm['session_id']}")
print(f"Domain hint: {wm['domain_hint']}")
print(f"History    : {len(wm['history'])} entries")
for i, h in enumerate(wm['history'], 1):
    print(f"\n  [{i}] EN: {h['source']}")
    print(f"       HI: {h['translation']}")
    print(f"       Score: {h['scores'].get('aggregate', 0):.3f}")

WORKING MEMORY — Session history
Session ID : 5ba65b45-efda-4307-a478-52697d5ea00a
Domain hint: None
History    : 6 entries

  [1] EN: Artificial intelligence is transforming the way we work and live.
       HI: कृत्रिम बुद्धिमत्ता हमारे काम करने और जीने के तरीके को बदल रही है।
       Score: 0.987

  [2] EN: Please submit your report by the end of the business day.
       HI: कृपया व्यावसायिक दिवस के अंत तक अपनी रिपोर्ट जमा करें।
       Score: 0.889

  [3] EN: The quick brown fox jumps over the lazy dog.
       HI: तेज़ भूरा लोमड़ी आलसी कुत्ते पर कूदता है।
       Score: 0.997

  [4] EN: Good morning! How was your weekend?
       HI: सुप्रभात! आपका सप्ताहांत कैसा रहा?
       Score: 0.992

  [5] EN: The meeting has been rescheduled to 3 PM tomorrow.
       HI: बैठक कल ३ बजे दोपहर के लिए पुनः निर्धारित की गई है।
       Score: 0.959

  [6] EN: Technology is making our lives easier every day.
       HI: प्रौद्योगिकी हमारे जीवन को हर दिन आसान बना रही है।
       Score: 0.989


In [ ]:
# ─── 9b. Episodic Memory ──────────────────────────────────────────────────────

print("=" * 60)
print("EPISODIC MEMORY — Retrieval demo")
print("=" * 60)

test_query = "The project deadline has been extended by two days."
test_emb   = encode_text(test_query)
hits       = episodic_memory.retrieve_similar(test_emb, top_k=3, min_score=0.3)

print(f"Query: '{test_query}'")
print(f"Total episodes stored: {episodic_memory.count()}")
print(f"Similar episodes found: {len(hits)}")
for h in hits:
    print(f"\n  similarity={h['_similarity']:.3f}")
    print(f"  EN: {h['source']}")
    print(f"  HI: {h['translation']}")

EPISODIC MEMORY — Retrieval demo
Query: 'The project deadline has been extended by two days.'
Total episodes stored: 6
Similar episodes found: 3

  similarity=0.415
  EN: The meeting has been rescheduled to 3 PM tomorrow.
  HI: बैठक कल ३ बजे दोपहर के लिए पुनः निर्धारित की गई है।

  similarity=0.387
  EN: Technology is making our lives easier every day.
  HI: प्रौद्योगिकी हमारे जीवन को हर दिन आसान बना रही है।

  similarity=0.359
  EN: Please submit your report by the end of the business day.
  HI: कृपया व्यावसायिक दिवस के अंत तक अपनी रिपोर्ट जमा करें।


In [ ]:
# ─── 9c. Semantic Memory ──────────────────────────────────────────────────────

print("=" * 60)
print("SEMANTIC MEMORY — Knowledge retrieval demo")
print("=" * 60)

test_query  = "How do I say please and thank you formally in Hindi?"
test_emb    = encode_text(test_query)
sem_hits    = semantic_memory.retrieve(test_emb, top_k=4)

print(f"Query: '{test_query}'")
print(f"Total semantic chunks: {semantic_memory.count()}")
print(f"Retrieved chunks: {len(sem_hits)}")
for i, h in enumerate(sem_hits, 1):
    print(f"\n  [{i}] distance={h['distance']:.3f} | type={h['metadata'].get('type','?')}")
    print(f"       {h['chunk'][:120]}..." if len(h['chunk']) > 120 else f"       {h['chunk']}")

SEMANTIC MEMORY — Knowledge retrieval demo
Query: 'How do I say please and thank you formally in Hindi?'
Total semantic chunks: 22
Retrieved chunks: 4

  [1] distance=0.382 | type=register
       In formal Hindi, use 'कृपया' for please, 'धन्यवाद' for thank you, 'क्षमा करें' for excuse me.

  [2] distance=0.487 | type=phrase
       Thank you = धन्यवाद (formal) or शुक्रिया (informal). You're welcome = आपका स्वागत है.

  [3] distance=0.516 | type=phrase
       How are you? = आप कैसे हैं? (formal) / तुम कैसे हो? (informal). I am fine = मैं ठीक हूँ.

  [4] distance=0.583 | type=translation_pair
       English: Good morning! How was your weekend?
Hindi: सुप्रभात! आपका सप्ताहांत कैसा रहा?


## Step 10 — Fine-Tuning Pipeline (QLoRA)

Fine-tune a causal LLM for En→Hi translation using LoRA adapters.

**Prerequisites:** GPU runtime + a training JSONL file.

**Recommended base models:**
- `AI4Bharat/indic-llm-13b` — purpose-built for Indic languages
- `meta-llama/Meta-Llama-3-8B` — strong multilingual
- `mistralai/Mistral-7B-v0.3` — efficient, good multilingual
- `google/gemma-2b` — lightweight (4GB VRAM)

**Training data format (JSONL):**
```json
{"instruction": "Translate to Hindi:", "input": "Hello", "output": "नमस्ते"}
```

**Recommended datasets:**
- [Samanantar](https://huggingface.co/datasets/ai4bharat/samanantar) — 49M en-hi pairs
- [IIT Bombay En-Hi](http://www.cfilt.iitb.ac.in/iitb_parallel/) — 1.6M curated pairs
- [FLORES-200](https://huggingface.co/datasets/facebook/flores) — 200-lang benchmark

In [ ]:
# ─── 10a. Create sample training data ─────────────────────────────────────────
# (Replace this with your real dataset for actual fine-tuning)

import json
from pathlib import Path

SAMPLE_TRAINING_DATA = [
    {"instruction": "Translate to Hindi:", "input": "Hello, how are you?",            "output": "नमस्ते, आप कैसे हैं?"},
    {"instruction": "Translate to Hindi:", "input": "Good morning!",                   "output": "शुभ प्रभात!"},
    {"instruction": "Translate to Hindi:", "input": "What is your name?",              "output": "आपका नाम क्या है?"},
    {"instruction": "Translate to Hindi:", "input": "I am going to the market.",       "output": "मैं बाज़ार जा रहा हूँ।"},
    {"instruction": "Translate to Hindi:", "input": "Please help me.",                 "output": "कृपया मेरी सहायता करें।"},
    {"instruction": "Translate to Hindi:", "input": "The weather is beautiful today.", "output": "आज मौसम बहुत सुंदर है।"},
    {"instruction": "Translate to Hindi:", "input": "I love reading books.",           "output": "मुझे किताबें पढ़ना पसंद है।"},
    {"instruction": "Translate to Hindi:", "input": "Thank you very much.",            "output": "बहुत-बहुत धन्यवाद।"},
    {"instruction": "Translate to Hindi:", "input": "Where is the nearest hospital?",  "output": "सबसे नज़दीकी अस्पताल कहाँ है?"},
    {"instruction": "Translate to Hindi:", "input": "The meeting starts at 9 AM.",     "output": "बैठक सुबह 9 बजे शुरू होती है।"},
]

DATA_PATH = DATA_DIR / "en_hi_training.jsonl"
with open(DATA_PATH, "w", encoding="utf-8") as f:
    for record in SAMPLE_TRAINING_DATA:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"✅ Sample training data written to {DATA_PATH}")
print(f"   {len(SAMPLE_TRAINING_DATA)} training examples")
print("\nFirst record preview:")
print(json.dumps(SAMPLE_TRAINING_DATA[0], ensure_ascii=False, indent=2))

✅ Sample training data written to /content/translation_data/en_hi_training.jsonl
   10 training examples

First record preview:
{
  "instruction": "Translate to Hindi:",
  "input": "Hello, how are you?",
  "output": "नमस्ते, आप कैसे हैं?"
}


In [ ]:
# ─── 10b. Prompt template + dataset formatter ─────────────────────────────────

TRAIN_PROMPT_TEMPLATE = (
    "Below is an instruction to translate text into Hindi.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)

INFERENCE_PROMPT_TEMPLATE = (
    "Below is an instruction to translate text into Hindi.\n\n"
    "### Instruction:\nTranslate to Hindi:\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)


def format_training_example(example: dict) -> dict:
    return {"text": TRAIN_PROMPT_TEMPLATE.format(**example)}


def load_training_dataset(path: str):
    from datasets import Dataset
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    dataset = Dataset.from_list(records)
    dataset = dataset.map(format_training_example, remove_columns=dataset.column_names)
    print(f"Loaded {len(dataset)} training examples")
    return dataset


# Preview formatted prompt
print("Sample formatted training prompt:")
print("-" * 60)
print(format_training_example(SAMPLE_TRAINING_DATA[0])["text"])

Sample formatted training prompt:
------------------------------------------------------------
Below is an instruction to translate text into Hindi.

### Instruction:
Translate to Hindi:

### Input:
Hello, how are you?

### Response:
नमस्ते, आप कैसे हैं?


In [ ]:
# ─── 10c. LoRA configuration ──────────────────────────────────────────────────

def get_lora_config():
    from peft import LoraConfig, TaskType
    return LoraConfig(
        r=16,                    # LoRA rank — higher = more parameters, better fit
        lora_alpha=32,           # scaling: effective LR = alpha / r
        target_modules=[
            "q_proj", "v_proj", "k_proj", "o_proj",   # attention
            "gate_proj", "up_proj", "down_proj",       # FFN
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )


def get_bnb_config():
    """4-bit NF4 quantisation config for QLoRA."""
    import torch
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )


print("✅ LoRA + QLoRA configs defined")
print("   rank=16, alpha=32, dropout=0.05")
print("   target: q/k/v/o_proj + gate/up/down_proj")

✅ LoRA + QLoRA configs defined
   rank=16, alpha=32, dropout=0.05
   target: q/k/v/o_proj + gate/up/down_proj


In [ ]:
# ─── 10d. Fine-tuning function ────────────────────────────────────────────────
# NOTE: Requires a GPU runtime and a base model on HuggingFace Hub.
# Comment out the actual call at the bottom if you only want to define the function.

def fine_tune_model(
    base_model: str,
    data_path: str,
    output_dir: str,
    epochs: int = 3,
    batch_size: int = 4,
    max_seq_length: int = 512,
    learning_rate: float = 2e-4,
) -> None:
    """
    QLoRA fine-tuning pipeline using HuggingFace TRL SFTTrainer.

    Args:
        base_model     : HuggingFace model ID (e.g. 'meta-llama/Meta-Llama-3-8B')
        data_path      : Path to JSONL training file
        output_dir     : Where to save LoRA adapter weights
        epochs         : Number of training epochs
        batch_size     : Per-device training batch size
        max_seq_length : Maximum token length per example
        learning_rate  : AdamW learning rate
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
    from peft import get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig

    print(f"Loading base model: {base_model} ...")

    # Tokeniser
    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Model with 4-bit quantisation
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=get_bnb_config(),
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )
    model.config.use_cache      = False
    model.config.pretraining_tp = 1
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, get_lora_config())
    model.print_trainable_parameters()

    # Dataset
    train_dataset = load_training_dataset(data_path)

    # Training config
    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=max(1, 16 // batch_size),
        optim="paged_adamw_32bit",
        learning_rate=learning_rate,
        weight_decay=0.001,
        fp16=True,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_strategy="epoch",
        group_by_length=True,
        max_seq_length=max_seq_length,
        dataset_text_field="text",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=training_args,
        tokenizer=tokenizer,
    )

    print("Starting fine-tuning...")
    trainer.train()

    final_path = f"{output_dir}/final_model"
    trainer.model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    print(f"✅ Fine-tuning complete! Adapter saved to: {final_path}")


# ── Uncomment and edit to run fine-tuning ─────────────────────────────────────
# fine_tune_model(
#     base_model  = "google/gemma-2b",           # small model, 4GB VRAM
#     data_path   = str(DATA_PATH),
#     output_dir  = "/content/hindi_lora",
#     epochs      = 3,
#     batch_size  = 4,
# )

print("✅ fine_tune_model() defined (uncomment the call above to run)")

✅ fine_tune_model() defined (uncomment the call above to run)


In [ ]:
# ─── 10e. Load fine-tuned model for inference ─────────────────────────────────

def load_fine_tuned_model(adapter_dir: str):
    """
    Load base model + LoRA adapter for inference.
    Returns (model, tokenizer).
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    with open(f"{adapter_dir}/adapter_config.json") as f:
        adapter_cfg = json.load(f)
    base_model = adapter_cfg["base_model_name_or_path"]

    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
    base = AutoModelForCausalLM.from_pretrained(
        base_model, torch_dtype=torch.float16, device_map="auto"
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    print(f"✅ Fine-tuned model loaded from {adapter_dir}")
    return model, tokenizer


def translate_with_fine_tuned(source: str, model, tokenizer, max_new_tokens: int = 256) -> str:
    """Run inference with locally loaded fine-tuned model."""
    import torch
    prompt = INFERENCE_PROMPT_TEMPLATE.format(input=source)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    marker = "### Response:\n"
    return full.split(marker)[-1].strip() if marker in full else full.strip()


# Usage after fine-tuning:
# ft_model, ft_tokenizer = load_fine_tuned_model("/content/hindi_lora/final_model")
# output = translate_with_fine_tuned("Hello world", ft_model, ft_tokenizer)
# print(output)

print("✅ Fine-tuned inference helpers defined")

✅ Fine-tuned inference helpers defined


## Step 11 — Evaluation & Comparison Report

Run a structured evaluation over multiple sentences and compare results.

In [ ]:
# ─── Benchmark evaluation ─────────────────────────────────────────────────────

BENCHMARK = [
    {
        "source":    "The doctor advised me to rest for a week.",
        "reference": "डॉक्टर ने मुझे एक हफ्ते तक आराम करने की सलाह दी।",
    },
    {
        "source":    "Children learn best through play and exploration.",
        "reference": "बच्चे खेल और अन्वेषण के माध्यम से सबसे अच्छा सीखते हैं।",
    },
    {
        "source":    "Please call me back when you are free.",
        "reference": "जब आप फ्री हों तो कृपया मुझे वापस कॉल करें।",
    },
]

print("Running benchmark evaluation...\n")
print(f"{'Source':<45} {'BLEU':>6} {'chrF':>6} {'BERT':>6} {'Agg':>6} {'Iters':>5}")
print("-" * 80)

all_results = []
for item in BENCHMARK:
    r = translate(
        source_text=item["source"],
        reference_translations=[item["reference"]],
        max_iterations=3,
        pass_threshold=0.60,
    )
    all_results.append(r)
    s = r["final_scores"]
    print(
        f"{item['source'][:43]:<45} "
        f"{s.get('bleu',0):6.1f} "
        f"{s.get('chrf',0):6.1f} "
        f"{s.get('bert_score_f1',0):6.3f} "
        f"{s.get('aggregate',0):6.3f} "
        f"{r['iterations']:5d}"
    )

# Averages
avg_agg  = sum(r["final_scores"].get("aggregate", 0)      for r in all_results) / len(all_results)
avg_bert = sum(r["final_scores"].get("bert_score_f1", 0)  for r in all_results) / len(all_results)
avg_iter = sum(r["iterations"]                             for r in all_results) / len(all_results)

print("-" * 80)
print(f"{'AVERAGES':<45} {'':>6} {'':>6} {avg_bert:6.3f} {avg_agg:6.3f} {avg_iter:5.1f}")
print(f"\n✅ Benchmark complete. Mean aggregate score: {avg_agg:.3f}")

Running benchmark evaluation...

Source                                          BLEU   chrF   BERT    Agg Iters
--------------------------------------------------------------------------------

[Node 1 — Input] 'The doctor advised me to rest for a week.'
[Node 2 — Embedding] dim=768 | episodic_hits=0 | semantic_hits=5
[Node 3 — Translation] Iteration 1: 'डॉक्टर ने मुझे एक सप्ताह के लिए आराम करने की सलाह दी।'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Node 4 — Evaluator] ✅ PASSED | agg=0.816 (threshold=0.6) | BLEU=53.1 chrF=73.9 BERT=0.945 COMET=0.966
[Node 5 — Feedback] Translation accepted. Aggregate=0.816 | BLEU=53.11 | chrF=73.93 | BERTScore=0.945
[Node 6 — Memory] All 3 tiers updated (quality pair added to semantic)
[Node 7 — Output] Final translation: 'डॉक्टर ने मुझे एक सप्ताह के लिए आराम करने की सलाह दी।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  53.11 │  73.93 │   0.9454  │ 0.9657  │  0.8161
The doctor advised me to rest for a week.       53.1   73.9  0.945  0.816     1

[Node 1 — Input] 'Children learn best through play and exploration.'
[Node 2 — Embedding] dim=768 | episodic_hits=0 | semantic_hits=5
[Node 3 — Translation] Iteration 1: 'बच्चे खेल और अन्वेषण के माध्यम से सबसे अच्छी तरह से सीखते है...'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Node 4 — Evaluator] ✅ PASSED | agg=0.879 (threshold=0.6) | BLEU=61.1 chrF=89.5 BERT=0.964 COMET=0.977
[Node 5 — Feedback] Translation accepted. Aggregate=0.879 | BLEU=61.15 | chrF=89.48 | BERTScore=0.964
[Node 6 — Memory] All 3 tiers updated (quality pair added to semantic)
[Node 7 — Output] Final translation: 'बच्चे खेल और अन्वेषण के माध्यम से सबसे अच्छी तरह से सीखते हैं।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  61.15 │  89.48 │   0.9637  │ 0.9774  │  0.8795
Children learn best through play and explor     61.1   89.5  0.964  0.879     1

[Node 1 — Input] 'Please call me back when you are free.'
[Node 2 — Embedding] dim=768 | episodic_hits=0 | semantic_hits=5
[Node 3 — Translation] Iteration 1: 'कृपया जब आप स्वतंत्र हों तो मुझे फोन करें।'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Node 4 — Evaluator] ✅ PASSED | agg=0.635 (threshold=0.6) | BLEU=13.9 chrF=36.6 BERT=0.902 COMET=0.980
[Node 5 — Feedback] Translation accepted. Aggregate=0.635 | BLEU=13.88 | chrF=36.57 | BERTScore=0.902
[Node 6 — Memory] Working + Episodic updated (aggregate 0.635 < 0.70 — skipping semantic)
[Node 7 — Output] Final translation: 'कृपया जब आप स्वतंत्र हों तो मुझे फोन करें।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  13.88 │  36.57 │   0.9019  │ 0.9800  │  0.6348
Please call me back when you are free.          13.9   36.6  0.902  0.635     1
--------------------------------------------------------------------------------
AVERAGES                                                     0.937  0.777   1.0

✅ Benchmark complete. Mean aggregate score: 0.777


In [ ]:
# ─── Detailed per-sentence report ─────────────────────────────────────────────

for i, (item, result) in enumerate(zip(BENCHMARK, all_results), 1):
    print(f"\n{'='*65}")
    print(f"Sentence {i}")
    print(f"{'='*65}")
    print(f"English  : {item['source']}")
    print(f"Reference: {item['reference']}")
    print(f"Output   : {result['final_translation']}")
    print(f"Feedback : {result['feedback'][:200]}..." if len(result['feedback']) > 200 else f"Feedback : {result['feedback']}")
    print(f"\nIteration score progression:")
    print(format_score_table(result['score_history']))


Sentence 1
English  : The doctor advised me to rest for a week.
Reference: डॉक्टर ने मुझे एक हफ्ते तक आराम करने की सलाह दी।
Output   : डॉक्टर ने मुझे एक सप्ताह के लिए आराम करने की सलाह दी।
Feedback : Translation accepted. Aggregate=0.816 | BLEU=53.11 | chrF=73.93 | BERTScore=0.945

Iteration score progression:
Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  53.11 │  73.93 │   0.9454  │ 0.9657  │  0.8161

Sentence 2
English  : Children learn best through play and exploration.
Reference: बच्चे खेल और अन्वेषण के माध्यम से सबसे अच्छा सीखते हैं।
Output   : बच्चे खेल और अन्वेषण के माध्यम से सबसे अच्छी तरह से सीखते हैं।
Feedback : Translation accepted. Aggregate=0.879 | BLEU=61.15 | chrF=89.48 | BERTScore=0.964

Iteration score progression:
Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  61.15 │  89.48 │   0.9637  │ 0.9774  │  0.8795

Sentence 3


## Step 12 — Custom Knowledge & Domain Adaptation

Add your own domain-specific knowledge to the semantic memory.

In [ ]:
# ─── Add custom domain knowledge ──────────────────────────────────────────────

# Legal domain
legal_chunks = [
    "Legal term 'contract' in Hindi = अनुबंध or संविदा. 'Agreement' = समझौता.",
    "'Plaintiff' = वादी, 'Defendant' = प्रतिवादी, 'Verdict' = फ़ैसला/निर्णय.",
    "'Court' = न्यायालय (formal) or अदालत. 'Judge' = न्यायाधीश. 'Law' = कानून/विधि.",
    "'Property' = संपत्ति. 'Rights' = अधिकार. 'Duty' = कर्तव्य. 'Justice' = न्याय.",
]

# Medical domain
medical_chunks = [
    "'Blood pressure' = रक्तचाप. 'Heart rate' = हृदय गति. 'Fever' = बुखार.",
    "'Surgery' = शल्य चिकित्सा. 'Prescription' = नुस्खा/प्रिस्क्रिप्शन. 'Diagnosis' = निदान.",
    "'Infection' = संक्रमण. 'Antibiotic' = एंटीबायोटिक. 'Vaccine' = टीका/वैक्सीन.",
]

semantic_memory.add_knowledge(
    chunks=legal_chunks + medical_chunks,
    metadatas=(
        [{"type": "legal"}] * len(legal_chunks) +
        [{"type": "medical"}] * len(medical_chunks)
    ),
)

print(f"✅ Domain knowledge added")
print(f"   Total semantic chunks now: {semantic_memory.count()}")

# Test: legal query
test_emb = encode_text("The plaintiff filed a lawsuit against the defendant.")
hits = semantic_memory.retrieve(test_emb, top_k=3)
print("\nTest retrieval for legal sentence:")
for h in hits:
    print(f"  [{h['metadata'].get('type','?')}] {h['chunk'][:80]}")

✅ Domain knowledge added
   Total semantic chunks now: 31

Test retrieval for legal sentence:
  [legal] 'Plaintiff' = वादी, 'Defendant' = प्रतिवादी, 'Verdict' = फ़ैसला/निर्णय.
  [legal] 'Court' = न्यायालय (formal) or अदालत. 'Judge' = न्यायाधीश. 'Law' = कानून/विधि.
  [medical] 'Infection' = संक्रमण. 'Antibiotic' = एंटीबायोटिक. 'Vaccine' = टीका/वैक्सीन.


## Step 13 — Interactive Translation

Enter any English sentence and translate it interactively.

In [ ]:
# ─── Interactive loop ─────────────────────────────────────────────────────────

print("English → Hindi Translation Agent")
print("Type 'quit' to exit, 'memory' to view session memory.")
print("="*60)

while True:
    user_input = input("\nEnglish: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("Session ended. Goodbye!")
        break
    if user_input.lower() == "memory":
        wm = working_memory.to_dict()
        print(f"\nWorking memory: {len(wm['history'])} entries")
        print(f"Episodic memory: {episodic_memory.count()} episodes")
        print(f"Semantic memory: {semantic_memory.count()} chunks")
        continue

    result = translate(
        source_text=user_input,
        max_iterations=3,
        pass_threshold=0.55,
    )

    print(f"\nHindi  : {result['final_translation']}")
    print(f"Score  : {result['final_scores'].get('aggregate', 0):.3f} (iters={result['iterations']})")

English → Hindi Translation Agent
Type 'quit' to exit, 'memory' to view session memory.

English: My name is Divyanshi.

[Node 1 — Input] 'My name is Divyanshi.'
[Node 2 — Embedding] dim=768 | episodic_hits=0 | semantic_hits=5
[Node 3 — Translation] Iteration 1: 'मेरा नाम दिव्यांशी है।'
[Node 4 — Evaluator] ✅ PASSED | agg=0.970 (threshold=0.55) | BLEU=0.0 chrF=0.0 BERT=1.000 COMET=0.940
[Node 5 — Feedback] Translation accepted. Aggregate=0.970 | BLEU=0.0 | chrF=0.0 | BERTScore=1.000
[Node 6 — Memory] All 3 tiers updated (quality pair added to semantic)
[Node 7 — Output] Final translation: 'मेरा नाम दिव्यांशी है।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │   0.00 │   0.00 │   1.0000  │ 0.9400  │  0.9700

Hindi  : मेरा नाम दिव्यांशी है।
Score  : 0.970 (iters=1)

English: 

English: exit
Session ended. Goodbye!


## Step 14 — Save & Export Results

Export your translation results and evaluation report to files.

In [ ]:
# ─── Export evaluation report ─────────────────────────────────────────────────

import json
from datetime import datetime

REPORT = {
    "timestamp": datetime.now().isoformat(),
    "session_id": SESSION_ID,
    "model": os.getenv("TRANSLATION_MODEL", "gpt-4o"),
    "pass_threshold": float(os.getenv("PASS_THRESHOLD", "0.60")),
    "memory_stats": {
        "working_memory_entries": len(working_memory.to_dict()["history"]),
        "episodic_memory_episodes": episodic_memory.count(),
        "semantic_memory_chunks": semantic_memory.count(),
    },
    "benchmark_results": [
        {
            "source":           item["source"],
            "reference":        item["reference"],
            "translation":      result["final_translation"],
            "final_scores":     result["final_scores"],
            "iterations":       result["iterations"],
            "score_history":    result["score_history"],
        }
        for item, result in zip(BENCHMARK, all_results)
    ],
}

REPORT_PATH = DATA_DIR / "evaluation_report.json"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(REPORT, f, ensure_ascii=False, indent=2)

print(f"✅ Report saved to {REPORT_PATH}")

# Download from Colab
try:
    from google.colab import files
    files.download(str(REPORT_PATH))
    print("✅ Report downloaded to your local machine")
except ImportError:
    print("(Not in Colab — file saved at:", REPORT_PATH, ")")

# Summary
print("\n" + "="*60)
print("SESSION SUMMARY")
print("="*60)
print(f"Sentences translated  : {episodic_memory.count()}")
print(f"Semantic chunks total : {semantic_memory.count()}")
print(f"Mean aggregate score  : {avg_agg:.3f}")
print(f"Mean iterations       : {avg_iter:.1f}")

✅ Report saved to /content/translation_data/evaluation_report.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Report downloaded to your local machine

SESSION SUMMARY
Sentences translated  : 10
Semantic chunks total : 32
Mean aggregate score  : 0.777
Mean iterations       : 1.0
